In [ ]:
# these should be the only 3 lines of code that you need to change
# after that you can run the next 7 cells

repo_path = 'C:\\Sources\\Repos\\EnterpriseCloud\\'
project_path = repo_path + 'dbt\\lz_to_ods\\'
model_path = repo_path + 'models\\activedirectory\\COMPUTERLIST.SQL'

In [ ]:
import json
import os
import re
import oyaml as yaml
import pandas as pd

db_mappings = {
    'ADMIN_DEV': 'ADMIN',
    'ADMIN_SIT': 'ADMIN',
    'ADMIN_QA': 'ADMIN',
    'LZ_DEV': 'LZ',
    'LZ_SIT': 'LZ',
    'LZ_QA': 'LZ',
    'LZ_PROD': 'LZ',
    'ODS_DEV': 'ODS',
    'ODS_SIT': 'ODS',
    'ODS_QA': 'ODS',
    'ODS': 'ODS',
    'COMMON_DEV': 'COMMON',
    'COMMON_SIT': 'COMMON',
    'COMMON_QA': 'COMMON',
    'COMMON': 'COMMON',
    'PUB_DEV': 'PUB',
    'PUB_SIT': 'PUB',
    'PUB_QA': 'PUB',
    'PUB': 'PUB',
    'EDW_DEV': 'EDW',
    'EDW_SIT': 'EDW',
    'EDW_QA': 'EDW',
    'EDW': 'EDW',
}

In [ ]:
new_project_path = repo_path + 'dbt_project\\'

with open(project_path + 'target\\manifest.json', 'r') as f:
    manifest = json.load(f)

In [ ]:
# goes through each model and pairs it with the new name convention <<DATABASE>>__<<SCHEMA>>__<<TABLE/VIEW>>

new_names = {}

for node in manifest['nodes'].values():
    db = node['database'].upper()
    db = db_mappings.get(db, db)
    schema = node['schema'].upper()
    table = node['alias'].upper()
    new_name = '__'.join([db, schema, table])
    new_names[node['name']] = new_name

In [ ]:
# stores list of sources without dbt models

file_path = 'C:\\Sources\\Repos\\EnterpriseCloud\\dbt_project\\models\\_sources_temp.yml'
with open(file_path, 'r') as f:
    yml = yaml.full_load(f.read())

temp_sources = [source['name'] + '__' + table['name'] for source in yml['sources'] for table in source['tables']]

In [ ]:
project_name = project_path.split('\\')[-2]
model_name = model_path.split('\\')[-1][:-4]
node_name = f'model.{project_name}.{model_name}'
node = manifest['nodes'][node_name]

In [ ]:
node_path = node['original_file_path']
node_db = node['database'].upper()
node_db = db_mappings.get(node_db, node_db)
node_schema = node['schema'].upper()

sql = node['raw_code']
new_sql = sql.replace('\r', '')

# find and tweak configuration options
config_pattern = '{{(?: |\n)*?config\(((?:.|\n)*?)\)(?:.|\n)*?}}'
match = re.search(config_pattern, new_sql)
if match:
    config_block = match.group()
    configs = match.groups()[0]
    configs = re.sub('\n', '', configs)
    new_configs = []
    for item in configs.split(','):
        # ignore transient config
        if not re.match('transient *= *false', item):
            # rename incremental materialization to custom name "tel_incremental"
            new_configs.append(re.sub("materialized *= *'incremental'", "materialized='incremental'", item))
    new_config_block = '{{ config(\n    ' + ',\n    '.join(new_configs) + '\n) }}'
    new_sql = new_sql.replace(config_block, new_config_block)

# rewrites source calls to refs if not an lz_to_ods model
if project_name != 'lz_to_ods':
    for source in node['sources']:
        source_name, source_node = source
        source_fqn = f'source.{project_name}.{source_name}.{source_node}'
        manifest_source = manifest['sources'][source_fqn]

        source_db = manifest_source['database']
        source_db = db_mappings.get(source_db, source_db)
        source_schema = manifest_source['schema'].upper()
        source_table = manifest_source['identifier'].upper()
        source_model_name = f'{source_db}__{source_schema}__{source_table}'
        if source_model_name in temp_sources:
            ref = f"{{{{ source('{source_db}__{source_schema}', '{source_table}') }}}}"
        else:
            ref = f"{{{{ ref('{source_model_name}') }}}}"
        pattern = f"""{{{{ *source *\\( *['"]{source_name}['"], *['"]{source_node}['"] *\\) *}}}}"""
        new_sql = re.sub(pattern, ref, new_sql)

# rewrites existing ref calls to use new name pattern
for ref in node['refs']:
    ref_model_name  = ref['name']
    new_ref_model_name = new_names.get(ref_model_name, ref_model_name)
    new_ref = "{{ ref('"+new_ref_model_name+"') }}"
    ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
    new_sql = re.sub(ref_regex, new_ref, new_sql)

# saves file with new name pattern in new project location
model = node['name']
if model not in new_names:
    print(f'{model} not found. Might be disabled')

else:
    folder = '\\'.join(node_path.split('\\')[:-1])
    new_model = new_names.get(model, model)
    new_folder = f'models\\{node_db.lower()}\\{node_schema.lower()}'
    if not os.path.isdir(new_project_path+new_folder):
        os.mkdir(new_project_path+new_folder)
    new_path = new_project_path+new_folder+'\\'+new_model+'.SQL'
    with open(new_path, 'w+', encoding='utf-8') as f:
        f.write(new_sql)

## <font color='red'>STOP</font> here if you are converting a single model

## Original mass update code

In [ ]:
project_name = 'common_to_pub'


project_path = f'C:\\Sources\\Repos\\EnterpriseCloud\\dbt\\{project_name}\\'
new_project_path = 'C:\\Sources\\Repos\\EnterpriseCloud\\dbt_project\\'

db_mappings = {
    'ADMIN_DEV': 'ADMIN',
    'ADMIN_SIT': 'ADMIN',
    'ADMIN_QA': 'ADMIN',
    'LZ_DEV': 'LZ',
    'LZ_SIT': 'LZ',
    'LZ_QA': 'LZ',
    'LZ_PROD': 'LZ',
    'ODS_DEV': 'ODS',
    'ODS_SIT': 'ODS',
    'ODS_QA': 'ODS',
    'ODS': 'ODS',
    'COMMON_DEV': 'COMMON',
    'COMMON_SIT': 'COMMON',
    'COMMON_QA': 'COMMON',
    'COMMON': 'COMMON',
    'PUB_DEV': 'PUB',
    'PUB_SIT': 'PUB',
    'PUB_QA': 'PUB',
    'PUB': 'PUB',
    'EDW_DEV': 'EDW',
    'EDW_SIT': 'EDW',
    'EDW_QA': 'EDW',
    'EDW': 'EDW',
}

with open(project_path+'target\\manifest.json', 'r') as f:
    manifest = json.load(f)

In [ ]:
# goes through each model and pairs it with the new name convention <<DATABASE>>__<<SCHEMA>>__<<TABLE/VIEW>>

new_names = {}

for node in manifest['nodes'].values():
    db = node['database'].upper()
    db = db_mappings.get(db, db)
    schema = node['schema'].upper()
    table = node['alias'].upper()
    new_name = '__'.join([db, schema, table])
    new_names[node['name']] = new_name

In [ ]:
# stores list of sources without dbt models

file_path = 'C:\\Sources\\Repos\\EnterpriseCloud\\dbt_project\\models\\_sources_temp.yml'
with open(file_path, 'r') as f:
    yml = yaml.full_load(f.read())

temp_sources = [source['name'] + '__' + table['name'] for source in yml['sources'] for table in source['tables']]

In [ ]:
last_node = None

# go through each of the existing models in the old project
for node in manifest['nodes'].values():    
    node_path = node['original_file_path']
    node_db = node['database'].upper()
    node_db = db_mappings.get(node_db, node_db)
    node_schema = node['schema'].upper()

    if node_schema not in ['META', 'DBT_TEST__AUDIT', 'DEV']:
        sql = node['raw_code']
        new_sql = sql.replace('\r', '')

        # find and tweak configuration options
        config_pattern = '{{(?: |\n)*?config\(((?:.|\n)*?)\)(?:.|\n)*?}}'
        match = re.search(config_pattern, new_sql)
        if match:
            config_block = match.group()
            configs = match.groups()[0]
            configs = re.sub('\n', '', configs)
            new_configs = []
            for item in configs.split(','):
                # ignore transient config
                if not re.match('transient *= *false', item):
                    # rename incremental materialization to custom name "tel_incremental"
                    new_configs.append(re.sub("materialized *= *'incremental'", "materialized='incremental'", item))
            new_config_block = '{{ config(\n    ' + ',\n    '.join(new_configs) + '\n) }}'
            new_sql = new_sql.replace(config_block, new_config_block)

        # rewrites source calls to refs if not an lz_to_ods model
        if project_name != 'lz_to_ods':
            for source in node['sources']:
                source_name, source_node = source
                source_fqn = f'source.{project_name}.{source_name}.{source_node}'
                manifest_source = manifest['sources'][source_fqn]

                source_db = manifest_source['database']
                source_db = db_mappings.get(source_db, source_db)
                source_schema = manifest_source['schema'].upper()
                source_table = manifest_source['identifier'].upper()
                source_model_name = f'{source_db}__{source_schema}__{source_table}'
                if source_model_name in temp_sources:
                    ref = f"{{{{ source('{source_db}__{source_schema}', '{source_table}') }}}}"
                else:
                    ref = f"{{{{ ref('{source_model_name}') }}}}"
                pattern = f"""{{{{ *source *\\( *['"]{source_name}['"], *['"]{source_node}['"] *\\) *}}}}"""
                new_sql = re.sub(pattern, ref, new_sql)
        
        # rewrites existing ref calls to use new name pattern
        for ref in node['refs']:
            ref_model_name  = ref['name']
            new_ref_model_name = new_names.get(ref_model_name, ref_model_name)
            new_ref = "{{ ref('"+new_ref_model_name+"') }}"
            ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
            new_sql = re.sub(ref_regex, new_ref, new_sql)
        
        # saves file with new name pattern in new project location
        model = node['name']
        if model not in new_names:
            print(f'{model} not found. Might be disabled')
            continue
        folder = '\\'.join(node_path.split('\\')[:-1])
        new_model = new_names.get(model, model)
        new_folder = f'models\\{node_db.lower()}\\{node_schema.lower()}'
        if not os.path.isdir(new_project_path+new_folder):
            os.mkdir(new_project_path+new_folder)
        new_path = new_project_path+new_folder+'\\'+new_model+'.SQL'
        with open(new_path, 'w+', encoding='utf-8') as f:
            f.write(new_sql)
        
        # rewrites yaml to match new models
        old_yml_path = project_path+folder+'\\schema.yml'
        new_sources_path = new_project_path+new_folder+'\\_sources.yml'
        new_models_path = new_project_path+new_folder+'\\_models.yml'
        
        # if schema.yml hasn't been converted already, split into sources and models using new naming convention
        if not os.path.exists(new_models_path): 
            new_models_yml = 'version: 2\n'
            if os.path.exists(old_yml_path):
                with open(old_yml_path, 'r', encoding='utf-8') as f:
                    yml = f.read()
            
                models_idx = yml.find('models:')
                if models_idx > 0:
                    sources_yml = yml[:models_idx]
                    models_yml = yml[models_idx:]
                else:
                    sources_yml = yml
                    models_yml = ''
                new_models_yml += models_yml

                if project_name == 'lz_to_ods':
                    with open(new_sources_path, 'w+', encoding='utf-8') as f:
                        f.write(sources_yml)
        else:
            with open(new_models_path, 'r', encoding='utf-8') as f:
                new_models_yml = f.read()
            
        if len(re.findall(f'\n *- *name: *{model}\\b', new_models_yml)) > 1:
            raise UserWarning(f'extra mentions of {model} in {new_models_path}')
        new_models_yml = re.sub(f'- *name: *{model}\\b.*', f'- name: {new_model}', new_models_yml)
        new_models_yml = re.sub('\n *identifier:.*', '', new_models_yml)

        with open(new_models_path, 'w+', encoding='utf-8') as f:
            f.write(new_models_yml)
        
        last_node = node
        # break
node = last_node


In [ ]:
# remove redundant models in yaml

root = 'C:\\Sources\\Repos\\EnterpriseCloud\\dbt_project\\models\\'

nodes = [name[:-4] for path, subdirs, files in os.walk(root) for name in files if name.endswith('.SQL')]

found = False
for path, subdirs, files in os.walk(root):
    for name in files:
        if name == '_models.yml':
            file_path = os.path.join(path, name)
            with open(file_path, 'r') as f:
                yml = yaml.full_load(f.read())

            if 'models' not in yml:
                continue

            new_models = []
            for model in yml['models']:
                node = model['name']
                if node in nodes:
                    new_models.append(model)
            yml['models'] = new_models

            with open(file_path, 'w') as f:
                f.write(yaml.dump(yml))


In [ ]:
# new project.yml schemas
project_yml = ''

dbs = ['ods', 'common', 'pub']
db = 'ods'

if db == 'pub':
    project_yml += ''

root_folder = f'models\\{db}\\'
for folder in os.listdir(new_project_path + root_folder):
    cleaned = False
    matched_models = []
    project_yml += f'      {folder}:\n        +schema: {folder.upper()}\n'
    if db == 'ods':
        project_yml += f'        +tags: {folder.lower()}\n'
print(project_yml)

In [ ]:
# find broken lineage

root = 'C:\\Sources\\Repos\\EnterpriseCloud\\dbt_project\\models\\'

nodes = [name[:-4] for path, subdirs, files in os.walk(root) for name in files if name.endswith('.SQL')]

for path, subdirs, files in os.walk(root):
    for name in files:
        file_path = os.path.join(path, name)
        try:
            with open(file_path, 'r') as f:
                sql = f.read()
            refs = re.findall("{{ *ref\\('(\\w+)'\\) *}}", sql)
            for ref in refs:
                if ref not in nodes:
                    print(ref)
        except:
            print('failed to process', file_path)